# HMRNet portable quickstart

Upload **HMRNet_Research_Code_v3.zip** in the first code cell. The second cell installs the core dependencies and runs the artificial-data functional test. The third connects Drive and runs final-model training using the original fixed feature bundle. This notebook does not publish files to GitHub.

Edit the feature path and choose a NEW output directory before the last cell. The final ten-seed run may take substantial time and RAM. For an initial run use `SEEDS = [1]`; this cannot estimate across-seed stability. The unchanged original setup notebook and the CLI `--dataset` mode can regenerate features if the historical feature folder is unavailable.


In [ ]:
from pathlib import Path
import zipfile
from google.colab import files
uploaded = files.upload()
archive = next(name for name in uploaded if name.endswith('.zip'))
base = Path('/content/HMRNet_v3_session')
if base.exists():
    raise FileExistsError('Use a new Colab runtime or choose a fresh extraction directory.')
base.mkdir()
with zipfile.ZipFile(archive) as z:
    for item in z.infolist():
        target = base / item.filename
        if not target.resolve().is_relative_to(base.resolve()):
            raise ValueError('Unsafe archive path')
        if (item.external_attr >> 16) & 0o170000 == 0o120000:
            raise ValueError('Archive symlinks are not accepted')
    z.extractall(base)
PROJECT_ROOT = base / 'HMRNet_Research_Code_v3'
assert (PROJECT_ROOT/'src/preprocessing.py').exists()
print(PROJECT_ROOT)


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(PROJECT_ROOT/'requirements.txt')], check=True)
subprocess.run([sys.executable, str(PROJECT_ROOT/'tests/smoke_recovered.py')], cwd=PROJECT_ROOT, check=True)
subprocess.run([sys.executable, str(PROJECT_ROOT/'scripts/check_reference.py')], cwd=PROJECT_ROOT, check=True)


In [ ]:
from google.colab import drive
from datetime import datetime

drive.mount('/content/drive')
FEATURES = Path('/content/drive/MyDrive/UDB/Riset/2026/results/checkpoints/features')
OUTPUT = Path('/content/drive/MyDrive/HMRNet_Reproduction') / datetime.now().strftime('%Y%m%d_%H%M%S')
SEEDS = list(range(1, 11))
assert FEATURES.exists(), 'Original feature folder not found; correct FEATURES first.'
command = [sys.executable, str(PROJECT_ROOT/'scripts/run_final_pipeline.py'),
           '--features', str(FEATURES), '--output', str(OUTPUT),
           '--seeds', *map(str, SEEDS)]
subprocess.run(command, cwd=PROJECT_ROOT, check=True)
print('New results:', OUTPUT)
